# Class 08 · Reproducible Reporting (Python)

Live-coding companion to the Week 4, Class B slides. Run the cells from top to bottom.
We build one Iris report step by step, render it from a fresh kernel, then add a parameter
to a second report and watch coverage change.

Keep this notebook in the folder that also holds `class-08-iris.ipynb`, `class-08-steam.ipynb` and
`steam_achievements.csv`.

In [ ]:
%matplotlib inline

## Part 1 · Why a dynamic document

- A result copied by hand into a report goes stale when the data change.
- A dynamic document keeps the question, code, evidence and interpretation in one source.
- Computed text updates a count after a rerun. An interpretation sentence still needs to be checked.
- A PDF records one run. The source still needs its input data and software to reproduce that run.

**Two separate stages.** Execution runs the code (the Jupyter kernel). Conversion turns the result into a
document (Pandoc, then XeLaTeX for PDF). The failing stage tells you which log to read.

## Part 2 · Build the Iris report, one cell at a time

**Question.** How do sepal lengths vary within and between the three species? Iris has 150 flowers, 50 per species, measured in centimetres.

### Setup: imports and data

In [ ]:
import platform
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.datasets import load_iris

iris = load_iris()
dat = pd.DataFrame(
    iris.data,
    columns=["Sepal_cm", "SepalWidth_cm", "Petal_cm", "PetalWidth_cm"],
)
dat["Species"] = iris.target_names[iris.target]
species = list(iris.target_names)

### Computed prose

A normal Markdown cell does not evaluate Python. This code cell formats its result as Markdown; `ddof=1` matches R's sample standard deviation.

In [ ]:
mean_sepal = dat["Sepal_cm"].mean()
sd_sepal = dat["Sepal_cm"].std(ddof=1)
display(
    Markdown(
        f"The mean sepal length is {mean_sepal:.2f} cm, "
        f"with sample standard deviation {sd_sepal:.3f} cm."
    )
)

> Check: mean 5.84 cm, sample SD 0.828 cm.

### A compact group summary

Group by species, count observations and compute the two means. The HTML and LaTeX forms are two representations of one calculation; each exporter picks the one it can use.

In [ ]:
summary_table = (
    dat.groupby("Species", sort=True)
    .agg(
        n=("Sepal_cm", "size"),
        Sepal_cm=("Sepal_cm", "mean"),
        Petal_cm=("Petal_cm", "mean"),
    )
    .reset_index()
)
display(
    {
        "text/html": summary_table.to_html(
            index=False, float_format="%.3f", escape=True
        ),
        "text/latex": summary_table.to_latex(
            index=False, float_format="%.3f", escape=True
        ),
    },
    raw=True,
)

> Every species has 50 flowers. Means alone hide the spread, so the report also needs plots.

### Figure 1: the pooled distribution

Record the bin grid in the report: 0.4 cm bins from 4.25 to 8.25 cm.

In [ ]:
edges = np.linspace(4.25, 8.25, 11)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(
    dat["Sepal_cm"],
    bins=edges,
    color="#D4A017",
    edgecolor="white",
)
ax.set(xlabel="Sepal length (cm)", ylabel="Number of flowers")
ax.set_title("Figure 1. Sepal length distribution", fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
np.histogram(dat["Sepal_cm"], bins=edges)[0]  # 9, 23, 20, 28, 19, 23, 16, 6, 5, 1 -> 150

### Figure 2: compare the species

Linear quartiles and 1.5-IQR whiskers. `figsize=(6, 3)` sets a six-by-three-inch figure; label the axes and call `tight_layout()`.

In [ ]:
groups = [dat.loc[dat["Species"].eq(name), "Sepal_cm"] for name in species]
fig, ax = plt.subplots(figsize=(6, 3))
ax.boxplot(
    groups,
    tick_labels=species,
    widths=0.55,
    patch_artist=True,
    boxprops={"facecolor": "#D4A017"},
    medianprops={"color": "black"},
)
ax.set(xlabel="Species", ylabel="Sepal length (cm)")
ax.set_title("Figure 2. Sepal lengths by species", fontsize=11)
fig.tight_layout()
plt.show()

> Setosa has the lowest median; the ranges overlap. The separate virginica point is 4.9 cm.

### Reproduction record

In [ ]:
print("Python", platform.python_version())
for package in [
    "pandas",
    "numpy",
    "matplotlib",
    "scikit-learn",
    "IPython",
    "nbconvert",
    "Jinja2",
]:
    print(package, version(package))

## Part 3 · Render from a fresh kernel

Saved outputs can come from an earlier, out-of-order run. `--execute` reruns every cell in a new kernel; `--no-input` hides the code but still runs it.

In [ ]:
import subprocess, sys

def render(notebook, *extra):
    cmd = [sys.executable, "-m", "jupyter", "nbconvert", "--execute",
           "--to", "pdf", "--no-input", *extra, notebook]
    done = subprocess.run(cmd, capture_output=True, text=True)
    print(done.stderr.strip().splitlines()[-1])

render("class-08-iris.ipynb")

## Part 4 · Check the result that will be submitted

1. Confirm the command completed without an execution error.
2. Open the new PDF and inspect every page.
3. Verify one count and one calculated value against the input.
4. Check every plot, unit, table and interpretation.
5. Preserve the source, the input identity and the software versions.

A successful render proves this configured run completed. It does not prove that the analysis
or the conclusion is correct.

## Part 5 · A parameterized report

A **parameter** names an input choice and records it. Changing it changes the selection while
the analytical recipe stays the same.

Data: six game records. A row is one game.

$$\text{completion rate} = \frac{\text{completed players}}{\text{listed players}}$$

- Completed players cannot exceed listed players.
- Players must be positive for a defined rate.
- The achievement count is **not** the rate denominator.

In [ ]:
print(open("steam_achievements.csv").read())

### Setup and an explicit parameter cell

In [ ]:
import hashlib
import platform
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

In [ ]:
datafile = "steam_achievements.csv"
minplayers = 100

### Validate, filter a copy, compute the rate

`raw` is kept unchanged; every assumption about the counts is checked.

In [ ]:
raw = pd.read_csv(datafile)
required = ["game", "players", "completed", "achievements"]
assert set(required) <= set(raw.columns)
assert not raw[required].isna().any().any()
counts = raw[["players", "completed", "achievements"]]
assert all(pd.api.types.is_numeric_dtype(counts[c]) for c in counts)
assert np.isfinite(counts.to_numpy()).all()
assert (counts >= 0).all().all()
assert (counts == np.floor(counts)).all().all()
assert (raw["players"] > 0).all()
assert (raw["completed"] <= raw["players"]).all()
assert raw["game"].is_unique and raw["game"].str.len().gt(0).all()
assert np.isfinite(minplayers) and minplayers >= 0
cleaned = raw.loc[raw["players"].ge(minplayers)].copy()
cleaned["rate"] = cleaned["completed"] / cleaned["players"]
cleaned = cleaned.sort_values(["rate", "game"], ascending=[False, True])

### Coverage, table and two figures

In [ ]:
display(
    Markdown(
        "## Coverage and comparison\n\n"
        f"The threshold is at least {minplayers} players per game. "
        f"This retains {len(cleaned)} of {len(raw)} games."
    )
)

In [ ]:
shown = cleaned[["game", "players", "completed", "achievements", "rate"]]
display(
    {
        "text/html": shown.to_html(
            index=False, float_format="%.3f", escape=True
        ),
        "text/latex": shown.to_latex(
            index=False, float_format="%.3f", escape=True
        ),
    },
    raw=True,
)

In [ ]:
if cleaned.empty:
    display(
        Markdown(
            "No games meet this threshold. There are no rates to plot."
        )
    )
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(cleaned["game"], cleaned["rate"], color="#D4A017")
    ax.invert_yaxis()
    ax.set(xlim=(0, 1), xlabel="Completed / players", ylabel="Game")
    ax.set_title("Figure 1. Completion rates", fontsize=11)
    fig.tight_layout()
    plt.show()

In [ ]:
if not cleaned.empty:
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(cleaned["game"], cleaned["players"], color="#777777")
    ax.invert_yaxis()
    ax.set(xlabel="Number of players", ylabel="Game")
    ax.set_title("Figure 2. Player counts", fontsize=11)
    fig.tight_layout()
    plt.show()

> At 100 players, four of six games remain. The rate plot compares proportions; the count plot shows how many players each proportion describes.

### Change the parameter

Predict first: how many games remain at 300? At 1000?

In [ ]:
for threshold in (100, 300, 1000):
    kept = raw.loc[raw["players"].ge(threshold)]
    rates = (kept["completed"] / kept["players"]).round(3).tolist()
    print(threshold, f"{len(kept)} of {len(raw)} games", rates)

> 300 keeps three games with the **same** per-game rates; filtering changes coverage, not the rate. 1000 keeps none: say so plainly instead of drawing an empty chart.

### Render one PDF per parameter value

Save a copy whose parameter cell says 300, then render it to its own file name.

In [ ]:
import nbformat

report = nbformat.read("class-08-steam.ipynb", as_version=4)
cell = next(c for c in report.cells if c.source.startswith("datafile"))
cell.source = cell.source.replace("minplayers = 100", "minplayers = 300")
nbformat.write(report, "report-300.ipynb")
render("report-300.ipynb")

## Part 6 · Troubleshooting and what to preserve

- Missing name or package: check setup and the selected environment.
- Missing file: check the working folder and the relative path.
- Pandoc error: check document conversion. XeLaTeX error: check the PDF compilation log.
- Preserve the source, raw input identity, parameters, software versions and the PDF from the final fresh run.